# Skill 42: Direct Preference Optimization (DPO)

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Compute the DPO loss from chosen/rejected log-probabilities of a policy and a frozen reference model.

## Objectives
- Understand chosen and rejected response pairs.
- Calculate policy-versus-reference log-probability differences.
- Implement the stable softplus form of the DPO objective.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$m=\beta\big[(\log\pi^w-\log\pi^l)-(\log\pi_{ref}^w-\log\pi_{ref}^l)\big],\qquad \mathcal L=\mathrm{softplus}(-m)=-\log\sigma(m)$$
When policy = reference, loss = ln 2 = 0.6931.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(policy_chosen, policy_rejected,
             reference_chosen, reference_rejected, beta=0.1):
    policy_margin = policy_chosen - policy_rejected
    reference_margin = reference_chosen - reference_rejected
    logits = beta * (policy_margin - reference_margin)
    return F.softplus(-logits).mean()

# Example batch: one log-probability per response.
pi_c = torch.tensor([-1.2, -0.8, -1.5], requires_grad=True)
pi_r = torch.tensor([-2.0, -1.7, -1.9], requires_grad=True)
ref_c = torch.tensor([-1.1, -1.0, -1.4])
ref_r = torch.tensor([-1.8, -1.6, -1.7])
loss = dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.1)
loss.backward()
print("DPO loss:", loss.item())
print("chosen policy gradients:", pi_c.grad)


## What to check
**Expected result:** Loss about 0.6849 for the sample batch; gradients finite.

## Pitfalls
Keep the reference model frozen. Sum log-probs over response tokens only. Do not swap chosen and rejected.